# Chatomics FigRemake - Log of Terminal Commands

### **Part 1**: Downloading the '.fastq.gz' files

In [ ]:
# Create conda env 'stdbio' with specific channels (conda-forge and bioconda) and the 'fastq-dl' module
! conda create -n stdbio -c conda-forge -c bioconda fastq-dl

# Activate the conda env
! conda activate stdbio

In [ ]:
# Each fastq file shall be placed in a 'fastq' folder
! mkdir fastq

# Download the data from the Gene Expression Omnibus (GEO) using 'fastq-dl'
# 'SRR1810900' is the SRR (ID tag) for the Sequence Read Archive (SRA)
! fastq-dl --accession SRR1810900 -o fastq/ # YAP_1

# Renames the download information file to have an identifiable name
! mv fastq/fastq-run-info.tsv fastq/SRR1810900-run-info.tsv

In [ ]:
# Install the 'fastqc' module (Quality Control)
! conda install bioconda::fastqc

# Quality reports will be stored in a 'qc_reports' folder
! mkdir qc_reports

# Run the quality analysis - outputs a .HTML file to report the data in the .zip file 
! fastqc fastq/SRR1810900.fastq.gz --outdir qc_reports

### **Part 2**: Alignment

In [ ]:
# Install the 'refgenie' module for downloading genome 'assets'
! conda install refgenie

# 
# NOTE: CAN ADD TO BASHRC THE EXPORT ONE
# NOTE: init makes the /data and /alias folders and the .yaml config file
! export REFGENIE='genome_config.yaml'
! refgenie init -c $REFGENIE

#
! refgenie pull hg38/fasta

In [ ]:
! conda install bowtie2 -c bioconda
# Took 34.5 mins
! bowtie2 -x GRCh38_noalt_as/GRCh38_noalt_as -U fastq/SRR1810900.fastq.gz -S fastq/SRR1810900.sam --threads 8  --no-mixed --no-discordant -k 1

In [ ]:
! conda install samtools -c bioconda
! samtools view -bS fastq/SRR1810900.sam > fastq/SRR1810900.bam

! rm fastq/SRR1810900.sam

! samtools sort -@ 4 -o fastq/SRR1810900.sorted.bam fastq/SRR1810900.bam

! rm fastq/SRR1810900.bam

! samtools index fastq/SRR1810900.sorted.bam

### **Side**: Sequentially run each (due to limited storage space)
The above code is repeated (using get_bam.sh) for:
- SRR1810907 (TAZ_1)
- SRR1810912 (IgG)
- SRR1810918 (TEAD4_1)

In [ ]:
"""
Assume everything module already installed:
 - fastq-dl
 - fastqc
 - bowtie2
 - samtools
Assume folders already created (list folders here!!!!)
 - fastq
 - qc_reports
Assume hg38 (Human Genome) already downloaded in 'alias' & 'data' folders using the 'refgenie' module
Assume bowtie2 indexer already downloaded in 'GRCh38_noalt_as' folder
"""

# Gives permission to run '.sh' file
! chmod 744 get_bam.sh

# Runs the terminal commands in cycle.sh
! ./get_bam.sh